In [1]:
import pandas as pd
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
import numpy as np

In [11]:
# 1. Load data
df = pd.read_csv('csv/new.csv')

# Display first few rows of raw data
print("First 5 rows of raw data:")
print(df.head())
print(f"\nDataset shape: {df.shape}")

# 2. Create Composition object column
# Note: Column names in CSV may contain special characters, let's check column names first
print("\nCSV column names:", df.columns.tolist())

# Assume the first column is the chemical formula, rename it to 'formula' for easier handling
df = df.rename(columns={df.columns[0]: 'formula'})
print("Renamed column names:", df.columns.tolist())

# Create Composition objects
print("\nStarting to create Composition objects...")
df['composition'] = df['formula'].apply(lambda x: Composition(x))

# Check if a few compositions were created successfully
print("\nComposition objects for first 3 materials:")
for i, (formula, comp) in enumerate(zip(df['formula'].head(3), df['composition'].head(3))):
    print(f"{formula} -> {comp}")

# 3. Initialize Magpie feature generator
print("\nInitializing Magpie feature generator...")
featurizer = ElementProperty.from_preset(preset_name="magpie")

# Check number of features
feature_labels = featurizer.feature_labels()
print(f"Will generate {len(feature_labels)} Magpie features")

# 4. Generate features
print("\nStarting Magpie feature generation...")
df_with_features = featurizer.featurize_dataframe(df, col_id='composition')
print("Feature generation completed!")

# Display feature dimensions
print(f"\nExtended dataset shape: {df_with_features.shape}")
print(f"Original number of features: {len(df.columns)}")
print(f"New Magpie features added: {len(feature_labels)}")

# View the newly added feature columns
magpie_columns = [col for col in df_with_features.columns if any(prefix in col for prefix in ['Magpie', 'mean', 'min', 'max', 'range', 'dev', 'mode'])]
print(f"\nFirst 20 Magpie feature column names:")
for i, col in enumerate(magpie_columns[:20]):
    print(f"{i+1:3d}. {col}")

# Save results to new_magpie.csv
output_file = 'csv/new_magpie.csv'
df_with_features.to_csv(output_file, index=False, encoding='utf-8')
print(f"\nResults saved to: {output_file}")

# Display some statistics
print("\nStatistical summary of selected features:")
stats_cols = ['formula', 'SúĘmuV/Kúę', 'PF(W/mK?)', 'ZT'] + magpie_columns[:5]
if all(col in df_with_features.columns for col in ['formula', 'SúĘmuV/Kúę', 'PF(W/mK?)', 'ZT']):
    stats_df = df_with_features[stats_cols].head()
    print(stats_df)

First 5 rows of raw data:
     formula  T(K)
0  Li2Pb2Se3   400
1  Li2Pb2Se3   800
2  Li2Pb2Se3   500
3       PbSe   300
4       PbSe   400

Dataset shape: (106, 2)

CSV column names: ['formula', 'T(K)']
Renamed column names: ['formula', 'T(K)']

Starting to create Composition objects...

Composition objects for first 3 materials:
Li2Pb2Se3 -> Li2 Pb2 Se3
Li2Pb2Se3 -> Li2 Pb2 Se3
Li2Pb2Se3 -> Li2 Pb2 Se3

Initializing Magpie feature generator...
Will generate 132 Magpie features

Starting Magpie feature generation...


C:\Users\wangshuaijie\PycharmProjects\JupyterProject\.venv\Lib\site-packages\matminer\utils\data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


ElementProperty:   0%|          | 0/106 [00:00<?, ?it/s]

Feature generation completed!

Extended dataset shape: (106, 135)
Original number of features: 3
New Magpie features added: 132

First 20 Magpie feature column names:
  1. MagpieData minimum Number
  2. MagpieData maximum Number
  3. MagpieData range Number
  4. MagpieData mean Number
  5. MagpieData avg_dev Number
  6. MagpieData mode Number
  7. MagpieData minimum MendeleevNumber
  8. MagpieData maximum MendeleevNumber
  9. MagpieData range MendeleevNumber
 10. MagpieData mean MendeleevNumber
 11. MagpieData avg_dev MendeleevNumber
 12. MagpieData mode MendeleevNumber
 13. MagpieData minimum AtomicWeight
 14. MagpieData maximum AtomicWeight
 15. MagpieData range AtomicWeight
 16. MagpieData mean AtomicWeight
 17. MagpieData avg_dev AtomicWeight
 18. MagpieData mode AtomicWeight
 19. MagpieData minimum MeltingT
 20. MagpieData maximum MeltingT

Results saved to: csv/new_magpie.csv

Statistical summary of selected features:


In [12]:
# Read CSV file
df = pd.read_csv('csv/new_magpie.csv')

# Delete the columns: S(muV/K), ¦Ò(S/cm), ZT
# Note: Column names might appear garbled due to encoding issues. Here we drop by name.
cols_to_drop = ['S(muV/K)', '¦Ò(S/cm)', 'ZT', 'PF']
df = df.drop(columns=cols_to_drop, errors='ignore')

# Rename TC(W/mK) to k(W/mK)
df = df.rename(columns={'TC(W/mK)': 'k(W/mK)'})

# Save to a new file
df.to_csv('csv/new_magpie.csv', index=False)